# 04 · ML Model Training — from split to saved model

**Round 1 demo:** this notebook exposes each training decision and every fit/score step. The model task is binary candidate ranking, because each trip has its own local arc IDs. Model selection is based on a whole-trip holdout from the 80 training trips. Student-test metrics are reported for development, not used to choose the model.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/"src").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
DATA=ROOT/"data"/"Student folder"
print("Project:",ROOT,"| dataset found:",DATA.exists())

Project: /Users/yashas/Documents/Codex/2026-11-18/referenced-chatgpt-conversation-this-is-an-2/ML_hackathon | dataset found: True


## Step 1 — Load partitions and generate the same candidate features for each model

In [2]:
import time,json,joblib
from pathsnap.data import read_trip
from pathsnap.matcher import generate_candidates,FEATURE_NAMES
train_trips=[read_trip(p,"train") for p in sorted((DATA/"train").iterdir()) if p.is_dir() and (p/f"{p.name}.track").exists()]
student_trips=[read_trip(p,"student_test") for p in sorted((DATA/"student_test").iterdir()) if p.is_dir() and (p/f"{p.name}.track").exists()]
print("Training trips:",len(train_trips),"| Student-test trips:",len(student_trips),"| k candidates:",12)
t0=time.perf_counter()
train_batches=[(t.trip_id,generate_candidates(t,k=12)) for t in train_trips]
student_batches=[(t.trip_id,generate_candidates(t,k=12)) for t in student_trips]
print("Candidate generation seconds:",round(time.perf_counter()-t0,2))
print("Example feature tensor:",train_batches[0][1].features.shape,"= observations × candidates × features")

Training trips: 80 | Student-test trips: 10 | k candidates: 12


Candidate generation seconds: 3.5
Example feature tensor: (1315, 12, 5) = observations × candidates × features


## Step 2 — Hold out whole trips, not random GPS rows

A random row split could put neighboring points from the same route into both training and validation. We use `GroupShuffleSplit` on trip IDs: 64 trips fit the initial models and 16 complete trips validate them.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
batch_groups=np.asarray([tid for tid,_ in train_batches])
splitter=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=42)
fit_idx,val_idx=next(splitter.split(np.arange(len(train_batches)),groups=batch_groups))
fit_trip_ids=batch_groups[fit_idx].tolist();validation_trip_ids=batch_groups[val_idx].tolist()
print("Fit trips:",len(fit_trip_ids),"Validation trips:",len(validation_trip_ids),"Overlap:",set(fit_trip_ids)&set(validation_trip_ids))

Fit trips: 64 Validation trips: 16 Overlap: set()


## Step 3 — Flatten candidate rows and construct the supervised target

For each GPS point, one candidate matching the nearest geometry in the reference route is positive. Other candidates are negative. If the reference road was missed by candidate generation, the observation is excluded from ranker fitting and counted separately; otherwise the model would have no positive option to learn from.

In [4]:
def flatten_batches(batches):
  xs=[];ys=[];groups=[]
  for trip_id,b in batches:
    eligible=(b.truth>=0)&(~b.candidate_miss)
    x=b.features[eligible].reshape(-1,len(FEATURE_NAMES))
    ids=b.ids[eligible].reshape(-1); truth=np.repeat(b.truth[eligible],b.ids.shape[1])
    xs.append(x);ys.append((ids==truth).astype(np.int8));groups.extend([trip_id]*len(x))
  return np.concatenate(xs),np.concatenate(ys),np.asarray(groups)
fit_batches=[train_batches[i] for i in fit_idx];val_batches=[train_batches[i] for i in val_idx]
X_raw,y_raw,groups=flatten_batches(fit_batches)
print("Before sampling:",X_raw.shape,"candidate rows | positive:",int(y_raw.sum()),"negative:",int((y_raw==0).sum()),"positive fraction:",round(float(y_raw.mean()),4))
print("Fit-only candidate misses:",sum(int(b.candidate_miss.sum()) for _,b in fit_batches))

Before sampling: (1793700, 5) candidate rows | positive: 149475 negative: 1644225 positive fraction: 0.0833
Fit-only candidate misses: 19482


## Step 4 — Handle the candidate class imbalance consistently

There are many more alternative roads than the single correct road. Keep all positive rows, sample at most four negatives per positive, then cap the first phase training sample at 200,000 rows with a fixed seed. Logistic Regression and KNN also standardize input scales; tree models use the original metre/radian features.

In [5]:
rng=np.random.default_rng(42)
pos=np.flatnonzero(y_raw==1);neg=np.flatnonzero(y_raw==0)
neg_keep=rng.choice(neg,size=min(len(neg),4*len(pos)),replace=False)
selected_rows=np.concatenate((pos,neg_keep))
if len(selected_rows)>200_000:selected_rows=rng.choice(selected_rows,size=200_000,replace=False)
X_train=X_raw[selected_rows];y_train=y_raw[selected_rows]
print({"training_rows_used":len(y_train),"positive_rows":int(y_train.sum()),"negative_rows":int((y_train==0).sum()),"positive_fraction":round(float(y_train.mean()),4),"seed":42})

{'training_rows_used': 200000, 'positive_rows': 39975, 'negative_rows': 160025, 'positive_fraction': 0.1999, 'seed': 42}


## Step 5 — Define five distinct model families and record settings

In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,HistGradientBoostingClassifier
models={
 "Logistic Regression":make_pipeline(StandardScaler(),LogisticRegression(max_iter=250,class_weight="balanced",random_state=42)),
 "KNN":make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=9,weights="distance",n_jobs=-1)),
 "Decision Tree":DecisionTreeClassifier(max_depth=14,min_samples_leaf=4,class_weight="balanced",random_state=42),
 "Random Forest":RandomForestClassifier(n_estimators=80,max_depth=18,min_samples_leaf=2,class_weight="balanced_subsample",n_jobs=-1,random_state=42),
 "Hist Gradient Boosting":HistGradientBoostingClassifier(max_iter=100,learning_rate=.08,max_leaf_nodes=15,l2_regularization=1.0,random_state=42)}
configs=pd.DataFrame([
 ["Logistic Regression","linear/probabilistic","StandardScaler; max_iter=250; balanced classes"],
 ["KNN","instance-based","StandardScaler; 9 neighbors; distance weights; balanced 5,000-row fit sample"],
 ["Decision Tree","tree","depth 14; leaf size 4; balanced classes"],
 ["Random Forest","bagged trees","80 trees; depth 18; leaf size 2; balanced subsampling"],
 ["Hist Gradient Boosting","boosted trees","100 iterations; learning rate .08; 15 leaves; L2=1.0"]],columns=["model","family","phase1_configuration"])
display(configs)

,model,family,phase1_configuration
0,Logistic Regression,linear/probabilistic,StandardScaler; max_iter=250; balanced classes
1,KNN,instance-based,StandardScaler; 9 neighbors; distance weights;...
2,Decision Tree,tree,depth 14; leaf size 4; balanced classes
3,Random Forest,bagged trees,80 trees; depth 18; leaf size 2; balanced subs...
4,Hist Gradient Boosting,boosted trees,100 iterations; learning rate .08; 15 leaves; ...


## Step 6 — Fit and score every model using the same splits

Top-1 is calculated by ranking candidates for each observation. Candidate Macro F1 and log loss are binary candidate-row metrics (reference candidate versus other candidate). Inference time covers scoring the student-test candidate tensors on this machine.

In [7]:
from sklearn.metrics import f1_score,log_loss
def score_batches(model,batches):
  top_correct=total=miss=0;true_rows=[];prob_rows=[];t0=time.perf_counter()
  for _,b in batches:
    n,k=b.ids.shape;prob=model.predict_proba(b.features.reshape(-1,len(FEATURE_NAMES)))[:,1].reshape(n,k)
    choice=np.argmax(prob,axis=1);pred=b.ids[np.arange(n),choice];valid=b.truth>=0
    top_correct+=int(np.sum((pred==b.truth)&valid));total+=int(valid.sum());miss+=int(np.sum(b.candidate_miss&valid))
    true_rows.append((b.ids[valid].reshape(-1)==np.repeat(b.truth[valid],k)).astype(np.int8));prob_rows.append(prob[valid].reshape(-1))
  elapsed=time.perf_counter()-t0;y=np.concatenate(true_rows);p=np.concatenate(prob_rows)
  return {"top1_accuracy":top_correct/total,"candidate_miss_rate":miss/total,"candidate_macro_f1":f1_score(y,(p>=.5).astype(int),average="macro",zero_division=0),"candidate_log_loss":log_loss(y,np.clip(p,1e-6,1-1e-6),labels=[0,1]),"inference_seconds":elapsed,"observations":total}
results=[];fit_models={}
for name,model in models.items():
  rows_for_fit=np.arange(len(y_train))
  if name=="KNN" and len(rows_for_fit)>5000:
    pidx=rows_for_fit[y_train==1];nidx=rows_for_fit[y_train==0]
    psel=rng.choice(pidx,size=min(len(pidx),2500),replace=False);nsel=rng.choice(nidx,size=min(len(nidx),5000-len(psel)),replace=False)
    rows_for_fit=np.concatenate([psel,nsel])
  t0=time.perf_counter();model.fit(X_train[rows_for_fit],y_train[rows_for_fit]);fit_seconds=time.perf_counter()-t0
  val_metrics=score_batches(model,val_batches);test_metrics=score_batches(model,student_batches)
  results.append({"model":name,"training_seconds":fit_seconds,**{f"validation_{k}":v for k,v in val_metrics.items()},**{f"student_test_{k}":v for k,v in test_metrics.items()}})
  fit_models[name]=model
  print(name,"fit seconds",round(fit_seconds,3),"validation Top-1",round(val_metrics["top1_accuracy"],4))
comparison=pd.DataFrame(results)
display(comparison[["model","training_seconds","validation_top1_accuracy","validation_candidate_macro_f1","student_test_top1_accuracy","student_test_candidate_macro_f1","student_test_candidate_log_loss","student_test_inference_seconds"]].style.format({c:"{:.4f}" for c in comparison.columns if c not in ["model"]}))

Logistic Regression fit seconds 0.05 validation Top-1 0.5522


KNN fit seconds 0.002 validation Top-1 0.6079


Decision Tree fit seconds 0.596 validation Top-1 0.6887


Random Forest fit seconds 2.359 validation Top-1 0.7109


Hist Gradient Boosting fit seconds 0.383 validation Top-1 0.7145


,model,training_seconds,validation_top1_accuracy,validation_candidate_macro_f1,student_test_top1_accuracy,student_test_candidate_macro_f1,student_test_candidate_log_loss,student_test_inference_seconds
0,Logistic Regression,0.0499,0.5522,0.6301,0.5650,0.6022,0.4722,0.0048
1,KNN,0.0023,0.6079,0.6965,0.6569,0.7251,0.5413,0.4895
2,Decision Tree,0.5960,0.6887,0.7989,0.7294,0.8024,0.2230,0.0074
3,Random Forest,2.3590,0.7109,0.8323,0.7514,0.8414,0.1385,0.1887
4,Hist Gradient Boosting,0.3834,0.7145,0.8478,0.7545,0.8609,0.1065,0.1176


## Step 7 — Compare with a non-learning nearest-road baseline

The baseline chooses the candidate with the smallest GPS-to-segment distance. It establishes whether learned ranking adds value over geometry alone.

In [8]:
def nearest_metrics(batches):
  correct=total=miss=0
  for _,b in batches:
    j=np.argmin(b.features[:,:,0],axis=1);pred=b.ids[np.arange(len(j)),j];valid=b.truth>=0
    correct+=int(np.sum((pred==b.truth)&valid));total+=int(valid.sum());miss+=int(np.sum(b.candidate_miss&valid))
  return {"top1_accuracy":correct/total,"candidate_miss_rate":miss/total,"observations":total}
print("Nearest road — validation:",nearest_metrics(val_batches))
print("Nearest road — student-test development:",nearest_metrics(student_batches))

Nearest road — validation: {'top1_accuracy': 0.5536746071971617, 'candidate_miss_rate': 0.07569690826153067, 'observations': 39460}
Nearest road — student-test development: {'top1_accuracy': 0.5347599164926932, 'candidate_miss_rate': 0.063517745302714, 'observations': 19160}


## Step 8 — Choose using validation, then refit on all training trips

Do not choose by the student-test score. Rank the five learned models by whole-trip validation Top-1; use the student-test table as a development-stage comparison only. Refit the selected model on candidates from all 80 training trips and save the artifact used by the local demo.

In [9]:
selected_name=max(results,key=lambda row:row["validation_top1_accuracy"])["model"]
print("Selected using validation Top-1 only:",selected_name)
X_all,y_all,_=flatten_batches(train_batches)
pos=np.flatnonzero(y_all==1);neg=np.flatnonzero(y_all==0)
neg_keep=rng.choice(neg,size=min(len(neg),4*len(pos)),replace=False)
all_rows=np.concatenate([pos,neg_keep])
if len(all_rows)>200_000:all_rows=rng.choice(all_rows,size=200_000,replace=False)
final_model=models[selected_name].__class__(**models[selected_name].get_params()) if False else fit_models[selected_name]
# The selected estimator above was fit only on the 64 fit trips; create a fresh instance and fit on all training trips.
from sklearn.base import clone
final_model=clone(models[selected_name])
final_fit=np.arange(len(all_rows))
if selected_name=="KNN" and len(final_fit)>5000:final_fit=rng.choice(final_fit,size=5000,replace=False)
final_model.fit(X_all[all_rows][final_fit],y_all[all_rows][final_fit])
import joblib
joblib.dump({"model":final_model,"model_name":selected_name,"feature_names":FEATURE_NAMES,"candidate_k":12,"seed":42},ROOT/"outputs"/"phase1_model.joblib")
comparison.to_csv(ROOT/"outputs"/"model_comparison.csv",index=False)
print("Saved all-training-trips model to outputs/phase1_model.joblib")

Selected using validation Top-1 only: Hist Gradient Boosting


Saved all-training-trips model to outputs/phase1_model.joblib


## Step 9 — Inspect model-driven feature importance on held-out trips

Permutation importance measures how much holdout Macro F1 changes when one feature is shuffled. This supports the feature discussion without using student-test labels for selection.

In [10]:
from sklearn.inspection import permutation_importance
X_val,y_val,_=flatten_batches(val_batches)
perm_rng=np.random.default_rng(42);sample=perm_rng.choice(len(y_val),size=min(12000,len(y_val)),replace=False)
importance=permutation_importance(fit_models[selected_name],X_val[sample],y_val[sample],scoring="f1_macro",n_repeats=3,random_state=42,n_jobs=-1)
importance_df=pd.DataFrame({"feature":FEATURE_NAMES,"mean_f1_drop":importance.importances_mean,"std":importance.importances_std}).sort_values("mean_f1_drop",ascending=False)
display(importance_df.round(4))
importance_df.to_csv(ROOT/"outputs"/"feature_importance.csv",index=False)

,feature,mean_f1_drop,std
2,along_segment_fraction,0.1977,0.0060
0,distance_m,0.1944,0.0042
1,heading_difference_rad,0.1166,0.0043
4,gps_speed_mps,0.0070,0.0014
3,segment_length_m,0.0054,0.0021


### Training decisions to explain

- The trip, not the GPS row, is the validation group; this reduces route leakage.
- Candidate misses are excluded from fit because there is no positive class in that candidate set.
- Negative candidates are sampled to make fitting tractable; the fixed seed makes the phase-1 run repeatable.
- Logistic Regression and KNN are scaled; tree/boosting models do not need feature scaling.
- The provisional model is selected from the training-trip validation score, then refit on all permitted training trips.
- The supplied student-test partition is development-only; evaluator-only trips were absent and untouched.